In [0]:
# Célula 1 — criar schema gold
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
print("Schema gold criado!")

Schema gold criado!


In [0]:
# Célula 2 — gold: orders_summary
gold_orders = spark.sql("""
    SELECT
        c.customer_state                                    AS estado,
        date_trunc('month', o.order_purchase_timestamp)    AS mes,
        COUNT(DISTINCT o.order_id)                         AS total_pedidos,
        ROUND(SUM(p.payment_value), 2)                     AS receita_total,
        ROUND(AVG(p.payment_value), 2)                     AS ticket_medio,
        ROUND(AVG(r.review_score), 2)                      AS nota_media
    FROM workspace.silver.olist_orders o
    JOIN workspace.silver.olist_customers  c USING (customer_id)
    JOIN workspace.silver.olist_payments   p USING (order_id)
    JOIN workspace.silver.olist_reviews    r USING (order_id)
    WHERE o.order_status = 'delivered'
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

gold_orders.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.orders_summary")

print(f"Gold orders_summary: {gold_orders.count()} linhas")
display(gold_orders.limit(5))

Gold orders_summary: 555 linhas


estado,mes,total_pedidos,receita_total,ticket_medio,nota_media
AC,2017-01-01T00:00:00.000Z,1,467.09,467.09,2.0
AC,2017-02-01T00:00:00.000Z,3,597.4,199.13,5.0
AC,2017-03-01T00:00:00.000Z,2,530.18,265.09,4.5
AC,2017-04-01T00:00:00.000Z,5,1351.51,270.3,3.2
AC,2017-05-01T00:00:00.000Z,8,2382.64,264.74,4.22


In [0]:
# Célula 3 — gold: customer_ltv
gold_ltv = spark.sql("""
    SELECT
        c.customer_state                        AS estado,
        COUNT(DISTINCT o.order_id)              AS total_pedidos,
        ROUND(SUM(p.payment_value), 2)          AS receita_total,
        ROUND(AVG(p.payment_value), 2)          AS ticket_medio,
        COUNT(DISTINCT c.customer_id)           AS total_clientes,
        ROUND(SUM(p.payment_value) / 
              COUNT(DISTINCT c.customer_id), 2) AS ltv_por_cliente
    FROM workspace.silver.olist_orders   o
    JOIN workspace.silver.olist_customers c USING (customer_id)
    JOIN workspace.silver.olist_payments  p USING (order_id)
    WHERE o.order_status = 'delivered'
    GROUP BY 1
    ORDER BY ltv_por_cliente DESC
""")

gold_ltv.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.customer_ltv")

print(f"Gold customer_ltv: {gold_ltv.count()} linhas")
display(gold_ltv)

Gold customer_ltv: 27 linhas


estado,total_pedidos,receita_total,ticket_medio,total_clientes,ltv_por_cliente
PB,517,137834.65,250.15,517,266.6
AC,80,19586.25,235.98,80,244.83
AP,67,16141.81,233.94,67,240.92
AL,397,94195.79,229.19,397,237.27
RO,243,56975.7,226.99,243,234.47
PA,946,212027.55,216.13,946,224.13
PI,476,105272.17,208.87,476,221.16
RR,41,9039.52,220.48,41,220.48
TO,274,60007.37,203.41,274,219.0
RN,474,100728.3,197.12,474,212.51


In [0]:
# Célula 4 — gold: delivery_performance
gold_delivery = spark.sql("""
    SELECT
        c.customer_state                                        AS estado,
        COUNT(DISTINCT o.order_id)                             AS total_pedidos,
        ROUND(AVG(
            datediff(o.order_delivered_customer_date,
                     o.order_purchase_timestamp)), 1)          AS prazo_medio_dias,
        ROUND(AVG(
            datediff(o.order_estimated_delivery_date,
                     o.order_delivered_customer_date)), 1)     AS antecipacao_media_dias,
        SUM(CASE WHEN o.order_delivered_customer_date 
                    > o.order_estimated_delivery_date 
                 THEN 1 ELSE 0 END)                            AS pedidos_atrasados,
        ROUND(AVG(r.review_score), 2)                          AS nps_medio
    FROM workspace.silver.olist_orders    o
    JOIN workspace.silver.olist_customers c USING (customer_id)
    JOIN workspace.silver.olist_reviews   r USING (order_id)
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY 1
    ORDER BY pedidos_atrasados DESC
""")

gold_delivery.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.delivery_performance")

print(f"Gold delivery_performance: {gold_delivery.count()} linhas")
display(gold_delivery)

Gold delivery_performance: 27 linhas


estado,total_pedidos,prazo_medio_dias,antecipacao_media_dias,pedidos_atrasados,nps_medio
SP,40068,8.7,11.1,2337,4.25
RJ,12146,15.2,11.8,1620,3.97
MG,11235,11.9,13.3,618,4.2
BA,3216,19.2,10.9,442,3.93
RS,5300,15.2,13.9,376,4.19
SC,3506,14.8,11.6,338,4.14
PR,4877,11.9,13.3,240,4.24
ES,1964,15.5,10.7,235,4.08
CE,1270,21.2,10.8,194,3.95
PE,1575,18.3,13.4,168,4.08


In [0]:
# Célula 5 — documentar tabelas Gold
spark.sql("""COMMENT ON TABLE workspace.gold.orders_summary IS 
    'Receita, ticket médio e NPS por estado e mês. Fonte: pedidos entregues.'""")

spark.sql("""COMMENT ON TABLE workspace.gold.customer_ltv IS 
    'LTV e ticket médio por estado. Fonte: pedidos entregues.'""")

spark.sql("""COMMENT ON TABLE workspace.gold.delivery_performance IS 
    'Prazo médio, atrasos e NPS por estado. Fonte: pedidos entregues com data de entrega.'""")

print("Tabelas Gold documentadas!")

Tabelas Gold documentadas!


In [0]:
# Célula 6 — Grant Select
spark.sql("""
    GRANT SELECT ON SCHEMA workspace.gold TO `account users`
""")
print("Grant Select Configurado")


configurado
